# A10 Master Notebook: Picking Winners Among US Theatrical Films

Shared, final notebook for the team submission (Matt Sorensen, Tanner Edwards). It follows **CRISP-DM**: each phase is a section, and every data decision is stated once with its evidence. Exploration and debate live in the personal notebooks (`Sorensen-A10.ipynb`, `Edwards-A10.ipynb`); only agreed work is promoted here (via PR).

**How to run (from a fresh clone)**

| Step | Command / notebook | Output |
|---|---|---|
| 1 | `python tools/get_data.py` | raw files in `data/raw/` (Wikidata snapshots are committed) |
| 2 | `notebooks/00_build_dataset.ipynb` (Run All) | `data/clean/films_unified.csv`, `reviews_unified.csv` |
| 3 | `notebooks/01_clean_dataset.ipynb` (Run All) | `data/clean/films_clean.csv`, `films_clean_columns.csv` |
| 4 | **this notebook** (Run All) | reads `films_clean.csv` only; writes nothing |

The pipeline notebooks do all merging and cleaning, so anyone can rebuild the data; this notebook shows *what* was done and *why*.

| CRISP-DM phase | Section |
|---|---|
| 1. Business Understanding | §1 |
| 2. Data Understanding (EDA) | §2 |
| 3. Data Preparation (cleaning) | §3 |
| 4. Modeling | §4 |
| 5. Evaluation | §5 (test set done; stress test and sensitivity to come) |
| 6. Deployment | §6 (to come) |

## 1. Business Understanding

From the Purpose Contract (`A10_purpose_contract_FROZEN_2026-11-17.md`) with amendments A1–A7, agreed by Tanner and Matt on 2026-11-24.

| Field | Answer |
|---|---|
| **Decision** (A6, A9) | Before a film's release, pick the films most likely to be **winners**; the rest get a closer look at their release and marketing plans before more money goes in. The screen never cancels or rejects a film on its own. |
| **Action** (A9) | Each year, the 20% of films most likely to be winners are identified. |
| **Inputs allowed** (A4, A6) | Anything the studio **controls before release**: budget, genre, runtime, release date, MPAA rating, director/writer/company track records (earlier films only), distributor, country, language, franchise, source material. Nothing known only after release (box office, reviews, scores, votes). |
| **Population** (A1, A2, A7) | Feature films **released in the US**, 2000–2023, budget ≥ $1M, after the cleaning rules in §3. Sensitivity rerun at a $100K floor. |
| **Label** (A3, A7, A9) | **Winner = US box office ≥ 1.25 × budget.** 1.25× US ≈ the industry rule of thumb of 2.5× worldwide (the US is about half of worldwide box office). It's a convention, not a data finding; rerun at 1× and 1.5×. |
| **Splits** (A5) | Train 2000–2015, test 2016–2019, stress test 2020–2023 (COVID era) reported separately. |
| **Baseline and bar** (A9) | Beat a genre × budget-tier rule by ≥ 5 percentage points in **winners caught in the top 20%**, with a bootstrap 95% CI excluding 0. |

**Framing (A9, agreed 2026-11-27):** the screen picks the 20% of films most likely to be **winners** (US box office ≥ 1.25× budget) each year.

**Amendments in one line each:** A9 pick the top 20% most likely winners · A1 keep 2000–2023 · A2 keep $1M floor, add $100K sensitivity · A3 2.5× is a convention (superseded by A7) · A4 inputs re-opened · A5 keep the COVID stress test · A6 timing rule = controlled before release · A7 label on US box office at 1.25×, population = released in the US.

## 2. Data Understanding

### 2a. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

CLEAN = Path("../data/clean")
films = pd.read_csv(CLEAN / "films_clean.csv", low_memory=False)
cols = pd.read_csv(CLEAN / "films_clean_columns.csv")
films["release_date"] = pd.to_datetime(films["release_date"], format="mixed", errors="coerce")

BLUE, ORANGE, GREEN, INK2 = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.titlesize": 10, "axes.titlelocation": "left",
                     "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8})

usable = films["usable"]
pop = films[films["in_population"]].copy()
pop["split"] = pd.cut(pop["year"], [1999, 2015, 2019, 2023], labels=["train 2000-15", "test 2016-19", "stress 2020-23"])
print(f"films_clean.csv: {len(films):,} films x {films.shape[1]} columns")
print(f"usable (2000-2023, budget >= $1M, revenue > 0): {usable.sum():,}   population (§3): {len(pop):,}")

### 2b. Sources and how they were joined

Six sources, assembled into one row per film by `00_build_dataset.ipynb` (key `film_uid`):

| Source | Adds | Join |
|---|---|---|
| TMDB (Kaggle, 1.5M rows) | base table, budget, revenue, genres, runtime, companies, countries, language, keywords | — |
| The Numbers (Kaggle scrape) | budget, **US and worldwide box office**, MPAA, source material, creative type, franchise | title + year matcher (hand-checked 50/50) + looser second pass |
| Box Office Mojo (GitHub scrape) | US box office, distributor | title + year |
| Rotten Tomatoes, two Kaggle datasets | director, writer, distributor, rating, critic reviews | title + year + runtime (hand-checked 0/50 wrong) |
| Wikidata snapshots (committed, CC0) | budgets; director, writer, runtime, "based on" | TMDB id |

How many outside sources (The Numbers, Box Office Mojo, the two RT datasets) confirm each usable film:

In [ ]:
src = films.loc[usable, "n_sources"].value_counts().sort_index()
display(pd.DataFrame({"usable films": src, "%": (100 * src / usable.sum()).round(1)}).rename_axis("outside sources").T)

**Finding:** 14% of usable films exist only in TMDB. TMDB is user-edited, and these are mostly junk entries (fan edits, TV episodes, a "$800M" 5-minute film) whose budget and revenue can't be checked. Handled in §3d.

### 2c. Missing values in the inputs (usable films, before filling)

The `_missing` flags record which inputs no source had, before any fill rule in §3.

In [ ]:
flags = [c for c in films.columns if c.endswith("_missing")]
miss = pd.DataFrame({"input": [c.removesuffix("_missing") for c in flags],
                     "% missing (usable)": [round(100 * films.loc[usable, c].mean(), 1) for c in flags],
                     "% missing (population)": [round(100 * pop[c].mean(), 1) for c in flags]})
display(miss.sort_values("% missing (usable)", ascending=False).set_index("input"))
tier = pd.cut(films["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
print("Missing rate by budget tier (usable films): coverage depends on budget, so blanks aren't random")
display(films.loc[usable, ["mpaa_rating_missing", "source_material_missing", "creative_type_missing", "distributor_missing"]]
        .groupby(tier[usable], observed=True).mean().mul(100).round(0))

### 2d. The inputs (population)

Distributions of each input on the final population (§3e), so the EDA describes the films the model will see.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
ax[0, 0].hist(np.log10(pop["fin_budget"]), bins=40, color=BLUE); ax[0, 0].set(title="Budget (log10 $; 7 = $10M)", ylabel="films")
ax[0, 1].hist(pop["runtime_min"].dropna(), bins=40, color=BLUE); ax[0, 1].set(title="Runtime (minutes)")
pop["release_date"].dt.month.value_counts().sort_index().plot.bar(ax=ax[0, 2], color=BLUE); ax[0, 2].set(title="Release month")
g = pop["genres_all"].dropna().str.split(",").explode().str.strip().value_counts()
g.sort_values().plot.barh(ax=ax[1, 0], color=BLUE); ax[1, 0].set(title="Genres (a film can have several)")
pop["mpaa_rating"].value_counts().plot.bar(ax=ax[1, 1], color=BLUE); ax[1, 1].set(title="MPAA rating (blank filled as NR)")
pop["source_group"].value_counts().plot.bar(ax=ax[1, 2], color=BLUE); ax[1, 2].set(title="Source material group")
for a in ax.flat:
    a.tick_params(axis="x", rotation=0)
ax[1, 1].tick_params(axis="x", rotation=0); ax[1, 2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

us = pop["countries"].fillna("").str.contains("United States")
print(f"Films including the US as a production country: {us.mean():.0%}; English-language: {pop['original_language'].eq('en').mean():.0%}; "
      f"in a franchise: {pop['franchise'].ne('(none)').mean():.0%}")

In [ ]:
def per_value(col, sep=","):
    v = pop[col].dropna().astype(str).str.split(sep).explode().str.strip()
    vc = v[v.ne("")].value_counts()
    b = pd.cut(vc, [0, 1, 4, 9, np.inf], labels=["1 film", "2-4", "5-9", "10+"])
    return b.value_counts().sort_index()

print("How many population films each person / company appears in (track records will be thin):")
display(pd.DataFrame({c: per_value(c) for c in ["director", "writer", "production_companies", "distributor"]}).T)

**Findings from the inputs** (detail: `Edwards-A10.ipynb` §6–7):
- **Budget** is very skewed (median about $15M, a long tail to $400M+); use it on a log scale or in tiers.
- **Runtime** is tight (middle half about 95–120 min) once non-films are removed (§3d).
- **Genres:** 19, mostly 2–3 per film; Drama and Comedy dominate.
- **People and companies:** most directors, writers and companies appear in only one film, so they work as experience counts or groups, not as names (see `FEATURE_IDEAS.md`).
- **Country and language** largely repeat each other; a US / non-US flag carries most of it.
- **Creative type and source material** are blank far more often for small films (The Numbers covers big films better).

### 2e. The label

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].hist(np.log10(pop["us_multiple"].clip(lower=1e-3)), bins=50, color=BLUE)
ax[0].axvline(np.log10(1.25), color=ORANGE, lw=1.5); ax[0].set(title="US box office / budget (log10; line = 1.25x)", ylabel="films")
by_year = pop.groupby("year")["winner"].mean()
ax[1].plot(by_year.index, 100 * by_year.values, color=BLUE, marker="o", ms=3); ax[1].set(title="Winner rate by year (%)", ylim=(0, 100))
ax[1].axvspan(2019.5, 2023.5, color=GREEN, alpha=.12)
tier_p = pd.cut(pop["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
(100 * pop.groupby(tier_p, observed=True)["winner"].mean()).plot.bar(ax=ax[2], color=BLUE); ax[2].set(title="Winner rate by budget tier (%)", ylim=(0, 100))
ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print(f"Winners: {pop['winner'].mean():.1%} at 1.25x; {pop['us_multiple'].ge(1).mean():.1%} at 1x; {pop['us_multiple'].ge(1.5).mean():.1%} at 1.5x")

### 2f. Data-quality problems found (and where they're handled)

| Problem | Evidence | Handled in |
|---|---|---|
| Junk entries that exist only in TMDB | 930 usable films in no outside source: fan edits, TV episodes, impossible budgets | §3d `tmdb_only` |
| Non-films: TV series, video games, TV movies, shorts, re-release editions | runtime 999 (*God of War*), TV Movie genre, "Extended Edition" titles | §3d |
| "Worldwide" revenue that is really the US gross | 618 films; only 2–11% labeled winners vs 30–38% | label switched to US box office (A7) |
| TMDB budget typos (extra zero) | 103 films where TMDB and The Numbers differ > 2× (*Awake* $86M vs $8.6M) | §3b |
| Same-title mix-ups | *A Christmas Carol* (2009) had a TV version's director; late-2023 films matched junk entries | build fix (late 2023) and §3c |
| Missing values depend on budget | creative type 64% blank for $1–5M films vs 9% for $60M+ | `_missing` flags, §3c fills |

### 2g. Each input vs the return (training years only)

Return = US box office ÷ budget; the label cuts it at 1.25×. Rows: population films from **2000–2015** (the 2016–19 test films stay unseen). Each row shows the group's median return (dot) and the middle half of its films (bar); dashed line = 1.25×. Full exploration: `Edwards-A10.ipynb` §10–16.

In [ ]:
trn = pop[pop["year"].le(2015)].copy()


def mult_plot(ax, groups, title, order=None, sort=False, min_n=30):
    t = trn.groupby(groups, observed=True)["us_multiple"].agg(films="size", med="median", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75))
    t = t[t["films"] >= min_n]
    if order is not None:
        t = t.reindex([o for o in order if o in t.index])
    if sort:
        t = t.sort_values("med", ascending=False)
    y = np.arange(len(t))[::-1]
    ax.hlines(y, t["q25"], t["q75"], color="#cde2fb", lw=7); ax.plot(t["med"], y, "o", color=BLUE, ms=6)
    ax.axvline(1.25, color="#9a9890", ls="--", lw=1)
    ax.set_xscale("log"); ax.set_xlim(0.02, 10)
    ax.set_xticks([0.05, 0.1, 0.25, 0.5, 1.25, 2, 4, 8], ["0.05x", "0.1x", "0.25x", "0.5x", "1.25x", "2x", "4x", "8x"]); ax.minorticks_off()
    ax.set_yticks(y, [f"{i}  (n={n:,})" for i, n in zip(t.index, t["films"])])
    ax.grid(False); ax.grid(axis="x", color="#e6e5e0", lw=0.6); ax.set_title(title)


fig, ax = plt.subplots(3, 2, figsize=(15, 12))
mult_plot(ax[0, 0], trn["mpaa_rating"].where(~trn["mpaa_rating_missing"].astype(bool), "blank (filled NR)"), "MPAA rating",
          order=["G", "PG", "PG-13", "R", "NR", "blank (filled NR)"])
mult_plot(ax[0, 1], trn["origin_group"], "Production origin", order=["US only", "US co-production", "non-US"])
mult_plot(ax[1, 0], trn["distributor_group"], "Distributor group", order=["Major", "Mini-major", "Other", "Specialty", "Blank"])
mult_plot(ax[1, 1], trn["sequel"].map({1: "sequel", 0: "not a sequel"}), "Sequel (franchise film already out, or installment in title)")
mult_plot(ax[2, 0], trn["source_group"].fillna("(blank)"), "Source material", sort=True)
mult_plot(ax[2, 1], trn["primary_genre"], "Primary genre", sort=True, min_n=40)
plt.tight_layout(); plt.show()
print(f"Training films: {len(trn):,}; median return {trn['us_multiple'].median():.2f}x; winners {trn['winner'].mean():.1%}")

**Findings from the inputs vs return** (detail and screening scores: `Edwards-A10.ipynb` §10–17)
- **Strongest signals:** distributor group (majors ≈ 1.2× vs specialty labels 0.02–0.3×), MPAA rating (G 1.31× → R 0.56× → NR 0.05×), origin (US-only 1.02× vs non-US 0.11×), sequel (3.25× the odds of winning after period and budget), company and director track records (share of earlier films that were hits).
- **Moderate:** primary genre (Family / Horror / Comedy ≈ 1× vs Drama / War / Western 0.3–0.5×), source material (adaptations ≈ 1.0× vs originals 0.67×; originals swing wider, mostly downward), competition from same/neighboring ratings within ±6 weeks, release weekday.
- **Budget is curved:** films under $10M usually lose in US theaters (median ≈ 0.25×); above $10M the median is flat near 0.8–1.0×. Bigger budgets make more dollars per film; smaller budgets more per dollar but about 70% lose.
- **Weak or none:** release season, tentpole flags, competition by budget, writer–director familiarity (it's really track record), genre count.
- **Warning:** inputs that are **blank** mark films the databases barely documented, which are mostly flops. The `_missing` flags would leak the outcome, so the model excludes them.
- **Trend context (all years, description only):** median return fell from 0.84× (2000–05) to about 0.65× (2006–15), recovered to 0.76× (2016–19), and dropped in COVID (0.59×). Sequels grew from 6% to 23% of releases and 15% to 55% of US box office. Horror rose (Blumhouse model); family films lost their edge.

## 3. Data Preparation

All steps run in `01_clean_dataset.ipynb`; this section reports what they did. Nothing is deleted: excluded films keep an `exclude_reason`, and `in_population` marks the model's films.

### 3a. One column per fact

Each fact came from several sources. Each gets one combined column, filled from the fullest source first; the source used is in `<column>_src`. Only real values count (TMDB's 0 = unknown).

In [ ]:
src_cols = ["runtime_min", "genres_all", "mpaa_rating", "director", "writer", "production_companies", "distributor",
            "countries", "original_language", "domestic_gross"]
rows = []
for c in src_cols:
    vc = pop[c + "_src"].value_counts(normalize=True)
    rows.append({"column": c, "% filled (population)": round(100 * pop[c].notna().mean(), 1),
                 "sources used (share of population)": ", ".join(f"{k} {v:.0%}" for k, v in vc.items())})
display(pd.DataFrame(rows).set_index("column"))

### 3b. Budget repair

When TMDB's budget and The Numbers' (or Wikidata's) differ by more than 2×, The Numbers' value is used.

In [ ]:
display(films.loc[usable, "fin_budget_src"].value_counts().rename("usable films").to_frame())

### 3c. Filling missing values

Decided input by input (Tanner, Edwards §9):
- **Director, writer, runtime:** filled from Wikidata (agrees with ours 96–97% where both exist). Where our director disagreed with Wikidata, ours came from a different film with the same title, so Wikidata's replaces it (`director_mismatch`).
- **Source material:** Wikidata "based on", then grouped into `source_group` (Comic / Book / Real life / Remake / Other adaptation / Original), filled last from TMDB keywords (74% agreement in a test; origin kept in `source_group_src`).
- **MPAA:** blank → NR. **Country:** blank → main country of the language. **Franchise:** blank → `(none)`.
- **Creative type, distributor:** left blank with a `_missing` flag.

In [ ]:
fill = {c: pop[c + "_src"].isin(["wikidata", "wikidata (replaced mismatch)"]).sum() for c in ["director", "writer", "runtime_min"]}
fill["source_material"] = pop["source_material_src"].eq("wikidata").sum()
display(pd.Series(fill, name="population films filled from Wikidata").to_frame())
print(f"Directors replaced because of a same-title mix-up: {pop['director_mismatch'].sum()}")
display(pd.crosstab(pop["source_group"].fillna("(blank)"), pop["source_group_src"].fillna("(blank)"), margins=True))

### 3d. Exclusion rules

Each usable film gets the first reason that applies (rules and examples: `Edwards-A10.ipynb` §8; definitions: `01_clean_dataset.ipynb` §12).

In [ ]:
ex = films.loc[usable & films["exclude_reason"].notna()]
summary = ex.groupby("exclude_reason").agg(films=("film_uid", "size"),
                                            examples=("title", lambda s: "; ".join(s.head(4))))
display(summary.sort_values("films", ascending=False))
funnel = pd.Series({"films in films_clean.csv": len(films), "usable (2000-23, budget >= $1M, revenue > 0)": usable.sum(),
                    "after exclusion rules": (usable & films["exclude_reason"].isna()).sum(),
                    "released in the US = population": len(pop)}, name="films")
display(funnel.to_frame())

### 3e. The prepared dataset

In [ ]:
split = pop.groupby("split", observed=True).agg(films=("film_uid", "size"), winners=("winner", "mean"))
split["winners"] = (100 * split["winners"]).round(1).astype(str) + "%"
display(split)
inputs = cols[cols["group"].eq("input") & ~cols["column"].str.endswith("_missing")]
display(inputs[["column", "% filled (population)"]].set_index("column").T)
print("Model inputs = the columns above, fin_budget, and the _missing flags. Label: winner (and us_multiple for sensitivity).")

### 3f. Engineered features (`01_clean_dataset.ipynb` §12b)

| Feature | What | Known before release because… |
|---|---|---|
| `log_budget` | log10 budget | budget is set before release |
| `primary_genre` | first genre listed | set at greenlight |
| `origin_group` | US only / US co-production / non-US | production countries |
| `sequel` | earlier franchise film already released, or installment number in the title (finds *Toy Story 3*, whose earlier films are pre-2000) | the film's own title and franchise |
| `distributor_group`, `major_producer` | Major / Mini-major / Specialty / Other / Blank; a major among the producers | distribution deal is signed before release (A6) |
| `release_weekday` | Wednesday / Friday / other | release date is set before release |
| `director_…`, `writer_…`, `company_prior_films` / `_prior_hit_rate` | number of earlier films, and the share that were winners | uses **earlier** films' results only |
| `rating_overlap_competition` | films in the same or neighboring rating (G↔PG↔PG-13↔R) within ±6 weeks, ÷ typical for the rating | competitors' release dates and ratings, never their box office |

In [ ]:
eng = ["log_budget", "primary_genre", "origin_group", "sequel", "distributor_group", "major_producer", "release_weekday",
       "director_prior_hit_rate", "writer_prior_hit_rate", "company_prior_hit_rate", "rating_overlap_competition"]
display(pd.DataFrame({"% filled (population)": (100 * pop[eng].notna().mean()).round(1),
                      "example": [pop[c].dropna().iloc[0] for c in eng]}).T)
print("Track-record blanks mean 'no earlier scored film' (first films, or careers before 2000), kept as their own group.")

## 4. Modeling

Promoted from `Edwards-A10.ipynb` §18–20 (Tanner, 2026-11-26 to 11-27), rewritten to run on its own from `films_clean.csv`. Same features, settings, folds and seeds; nothing retuned.

**Framing (A9):** a **winner** = US box office ≥ 1.25× budget (`winner`). The model ranks films by their chance of being a winner; the screen picks the **top 20%**.

**Everything in §4 is 5-fold cross-validation inside the training years (2000–2015).** The 2016–19 test set is used once, in §5.

**Measures**
- **Recall @20%** (winners caught) = share of all winners that land in the top 20% picks. With 33% winners, random picks catch 20%; the best possible is 60%.
- **Precision @20%** (hit rate) = share of the picks that are winners (random: 33%).
- **AUC** = ranking quality over all films (0.5 = random, 1 = perfect).

**Ties:** picks use a stable sort (`kind="stable"`), so films with equal scores are always broken in the same order on every machine. Tanner's original used NumPy's default sort, which can break ties differently across machines. Only the baseline (many films share its group scores) can move, by about one film; model scores have no ties. Changed by Matt, 2026-11-28, before rerunning; not tuning.

### 4a. Setup

In [ ]:
import warnings
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

TRN = films[films["in_population"] & films["year"].le(2015)].reset_index(drop=True)
yl = TRN["winner"].astype(int).values
folds = list(StratifiedKFold(5, shuffle=True, random_state=560).split(TRN, yl))
TIERS = ([0, 5e6, 20e6, 60e6, np.inf], ["$1-5M", "$5-20M", "$20-60M", "$60M+"])
print(f"Training films (2000-15): {len(TRN):,}; winners {yl.mean():.1%}; random picks catch 20% of winners, "
      f"best possible {min(1, 0.2 / yl.mean()):.0%}")


def top_k(score, k=0.2):
    """Indices of the top k share of films by score; stable sort so ties break the same way everywhere."""
    return np.argsort(-score, kind="stable")[:int(round(k * len(score)))]


def capture_at(score, y, k=0.2):
    top = top_k(score, k)
    return y[top].sum() / y.sum(), y[top].mean()


def cv_predict(model, X):
    oof = np.zeros(len(X))
    for tr_i, te_i in folds:
        model.fit(X.iloc[tr_i], yl[tr_i])
        oof[te_i] = model.predict_proba(X.iloc[te_i])[:, 1]
    return oof


def auc_range(s, y=yl, n=300, seed=560):
    rng = np.random.default_rng(seed)
    b = [roc_auc_score(y[i], s[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return f"{np.percentile(b, 2.5):.3f}-{np.percentile(b, 97.5):.3f}"


def summary_row(name, s, with_logloss=True):
    cap, prec = capture_at(s, yl)
    return {"model": name, "AUC": round(roc_auc_score(yl, s), 3), "AUC 95% range": auc_range(s),
            "recall @20%": f"{cap:.1%}", "precision @20%": f"{prec:.1%}",
            "log loss": round(log_loss(yl, np.clip(s, 1e-6, 1 - 1e-6)), 3) if with_logloss else np.nan}

### 4b. Baseline (b): genre × budget tier

The bar to beat (contract, A9): each film gets the historical winner rate of its genre × budget-tier group, learned on the training folds only. Thin groups are pulled toward the average (20 pseudo-films).

In [ ]:
g_b = TRN["primary_genre"].fillna("(blank)") + " | " + pd.cut(TRN["fin_budget"], *TIERS).astype(str)
oof_b = np.zeros(len(TRN))
for tr_i, te_i in folds:
    rates = pd.Series(yl[tr_i]).groupby(g_b.iloc[tr_i].values).agg(["sum", "size"])
    smooth = (rates["sum"] + 20 * yl[tr_i].mean()) / (rates["size"] + 20)
    oof_b[te_i] = g_b.iloc[te_i].map(smooth).fillna(yl[tr_i].mean()).values
rng = np.random.default_rng(560)
display(pd.DataFrame([summary_row("random 20%", rng.random(len(TRN)), False),
                      summary_row("baseline (b): genre x budget tier", oof_b, False)]).set_index("model").drop(columns="log loss"))

### 4c. Five model types on the same features

All inputs known before release (A6), **no `_missing` flags** (they leak the outcome: undocumented films are mostly flops; with them, AUC rises by about 0.04 for the wrong reason, `Edwards-A10.ipynb` §18). Same folds for every model.

| Model | In plain words |
|---|---|
| Logistic regression | one weight per feature; budget gets a curve (spline) |
| Decision tree | a single shallow tree, readable as if/then rules (depth 4) |
| Random forest | 500 trees averaged |
| Gradient boosting (scikit-learn) | trees built one after another, each fixing the last one's errors |
| XGBoost | the industry-standard version of gradient boosting |

In [ ]:
D = TRN.copy()
D["mpaa_rating"] = D["mpaa_rating"].replace({"NC-17": "R"})                       # 9 NC-17 films merged into R
D["primary_genre"] = D["primary_genre"].where(D["primary_genre"].map(D["primary_genre"].value_counts()) >= 40, "Other")
D["original_language"] = D["original_language"].where(D["original_language"].isin(D["original_language"].value_counts().head(5).index), "other")
for c in ["director", "writer", "company"]:
    D[f"{c}_has_history"] = D[f"{c}_prior_hit_rate"].notna().astype(int)          # 'no earlier film' is its own signal
CATS = ["mpaa_rating", "primary_genre", "origin_group", "distributor_group", "release_weekday", "source_group", "original_language"]
NUMS = ["runtime_min", "release_month", "sequel", "major_producer", "director_prior_films", "director_prior_hit_rate", "director_has_history",
        "writer_prior_films", "writer_prior_hit_rate", "writer_has_history", "company_prior_films", "company_prior_hit_rate", "company_has_history",
        "rating_overlap_competition"]
for c in CATS:
    D[c] = D[c].astype(object).fillna("(blank)").astype(str)
Xall = D[["log_budget"] + NUMS + CATS]

pre_linear = ColumnTransformer([
    ("budget", make_pipeline(SimpleImputer(strategy="median"), SplineTransformer(n_knots=5, degree=3)), ["log_budget"]),
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATS)])
pre_tree = ColumnTransformer([("num", "passthrough", ["log_budget"] + NUMS),
                              ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False), CATS)])


def xgb():
    return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                         min_child_weight=5, reg_lambda=1.0, eval_metric="logloss", random_state=560, n_jobs=4)


MODELS = {
    "Logistic regression": make_pipeline(pre_linear, LogisticRegression(max_iter=3000, C=0.5)),
    "Decision tree": make_pipeline(pre_tree, SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=560)),
    "Random forest": make_pipeline(pre_tree, SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=500, min_samples_leaf=20, max_features="sqrt", n_jobs=-1, random_state=560)),
    "XGBoost": make_pipeline(pre_tree, xgb()),
}
OOF = {name: cv_predict(m, Xall) for name, m in MODELS.items()}

# Gradient boosting (scikit-learn) uses native categories (Edwards §18 "cleanest" version: no _missing flags, distributor Blank = unknown)
Xhgb = TRN[["log_budget"] + [c for c in NUMS if not c.endswith("_has_history")]].astype(float).copy()
for c in CATS:
    s = TRN[c].astype(object)
    if c == "distributor_group":
        s = s.where(s.ne("Blank"))
    if c == "original_language":
        s = s.where(s.isin(s.value_counts().head(8).index), "other")
    Xhgb[c] = s.astype("category")
OOF["Gradient boosting (sklearn)"] = cv_predict(
    HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
                                   categorical_features="from_dtype", random_state=560), Xhgb)

cmp = pd.DataFrame([summary_row("Baseline (b)", oof_b, False)] + [summary_row(k, v) for k, v in OOF.items()]).set_index("model")
display(cmp.sort_values("AUC", ascending=False))

### 4d. Statistical fit of each feature (logistic regression, all training films)

- **McFadden pseudo-R²:** 1 − (log-likelihood of the model ÷ that of a model with no features). 0.2–0.4 is a good fit for logistic regression.
- **AIC:** fit penalized for the number of parameters; lower is better. A feature earns its place if removing it **raises** AIC by more than about 2.
- **Likelihood-ratio test:** refit without the feature (all its categories at once); p < 0.05 = keep.

Budget enters as log budget and its square (its curve). Numbers are standardized; categories are compared with PG-13 / Drama / US only / Major / Friday / Original / English.

In [ ]:
REF = {"mpaa_rating": "PG-13", "primary_genre": "Drama", "origin_group": "US only", "distributor_group": "Major", "release_weekday": "Friday",
       "source_group": "Original", "original_language": "en"}
LABEL = {"mpaa_rating": "MPAA rating", "primary_genre": "primary genre", "origin_group": "origin (US / non-US)", "distributor_group": "distributor group",
         "release_weekday": "release weekday", "source_group": "source group", "original_language": "original language"}
NUM_GROUPS = {"budget (log + curve)": ["log_budget_c", "log_budget_sq"], "runtime": ["runtime_min"], "release month": ["release_month"],
              "sequel": ["sequel"], "major producer": ["major_producer"],
              "director track record": ["director_prior_films", "director_prior_hit_rate", "director_has_history"],
              "writer track record": ["writer_prior_films", "writer_prior_hit_rate", "writer_has_history"],
              "company track record": ["company_prior_films", "company_prior_hit_rate", "company_has_history"],
              "rating-overlap competition": ["rating_overlap_competition"]}


def design(X, nums, cats):
    """statsmodels design matrix: standardized numbers (median-filled), budget curve, dummies vs reference."""
    S = X.copy()
    S["log_budget_c"] = S["log_budget"] - S["log_budget"].mean()
    S["log_budget_sq"] = S["log_budget_c"] ** 2
    terms = ["log_budget_c", "log_budget_sq"] + nums
    for c in terms:
        v = S[c].astype(float).fillna(S[c].astype(float).median())
        S[c] = (v - v.mean()) / v.std()
    dums = []
    for c in cats:
        dm = pd.get_dummies(S[c], prefix=c, dtype=float)
        dums.append(dm.drop(columns=f"{c}_{REF[c]}" if f"{c}_{REF[c]}" in dm else dm.columns[0]))
    XS = sm.add_constant(pd.concat([S[terms]] + dums, axis=1))
    groups = {g: cols for g, cols in NUM_GROUPS.items() if all(c in XS for c in cols)}
    groups.update({LABEL[c]: [x for x in XS.columns if x.startswith(c + "_")] for c in cats})
    return XS, groups


def logit(XS, cols):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return sm.Logit(yl, XS[cols]).fit(disp=0, maxiter=500, method="newton")


def fit_table(XS, groups):
    full = logit(XS, list(XS.columns))
    rows = []
    for g, cols in groups.items():
        red = logit(XS, [c for c in XS.columns if c not in cols]); lr = 2 * (full.llf - red.llf)
        rows.append({"feature": g, "terms": len(cols), "LR chi2": round(lr, 1), "p": chi2.sf(lr, len(cols)), "AIC if removed (+ = worse)": round(red.aic - full.aic, 1)})
    t = pd.DataFrame(rows).set_index("feature").sort_values("LR chi2", ascending=False)
    t["verdict"] = np.where((t["p"] < 0.05) & (t["AIC if removed (+ = worse)"] > 2), "keep", np.where(t["p"] < 0.10, "borderline", "drop candidate"))
    t["p-value"] = t.pop("p").map(lambda p: f"{p:.1e}" if p < 0.001 else f"{p:.3f}")
    return full, t


XS, G = design(D, NUMS, CATS)
null_ll = sm.Logit(yl, XS[["const"]]).fit(disp=0).llf
full, ft = fit_table(XS, G)
print(f"All candidate features: {len(XS.columns) - 1} terms; McFadden pseudo-R2 {1 - full.llf / null_ll:.3f}; AIC {full.aic:.1f}")
display(ft)

### 4e. Blank categories leak the outcome too

A "(blank)" source or origin marks a film the databases barely documented; in the training years every such film is a loser (perfect separation in 4d). That is the same leak as the `_missing` flags, hiding inside the categories. **Fix:** merge each blank into the most common category (distributor "Blank" → "Other") so the model can't use "undocumented" as a signal.

In [ ]:
Xclean = Xall.copy()
for c in CATS:
    Xclean[c] = Xclean[c].replace("(blank)", Xclean.loc[Xclean[c] != "(blank)", c].mode()[0])
Xclean["distributor_group"] = Xclean["distributor_group"].replace("Blank", "Other")
res = []
for name in ["Logistic regression", "XGBoost", "Random forest"]:
    o = cv_predict(MODELS[name], Xclean); cap, prec = capture_at(o, yl)
    res.append({"model": name, "AUC with blank categories": round(roc_auc_score(yl, OOF[name]), 3), "AUC blanks merged": round(roc_auc_score(yl, o), 3),
                "recall @20% (merged)": f"{cap:.1%}", "precision @20% (merged)": f"{prec:.1%}"})
display(pd.DataFrame(res).set_index("model"))
print("Training films with any blank category:", int(((Xall[CATS] == "(blank)").any(axis=1) | (Xall["distributor_group"] == "Blank")).sum()))

### 4f. Final model

Agreed feature set (Tanner, 2026-11-27, `Edwards-A10.ipynb` §20):
- **Dropped:** release month, writer track record, original language (each failed the likelihood-ratio test in 4d and *lowered* AIC when removed); source group (failed once blanks were merged, `Edwards-A10.ipynb` §20b: p = 0.39, AIC −4.8 when removed; genre, rating and sequel cover it).
- **Kept as borderline:** director track record, major producer.
- **Blanks merged** as in 4e. **Main model:** logistic regression. **Challenger:** XGBoost, same features.

In [ ]:
NUMS_F = ["runtime_min", "sequel", "major_producer", "director_prior_films", "director_prior_hit_rate", "director_has_history",
          "company_prior_films", "company_prior_hit_rate", "company_has_history", "rating_overlap_competition"]
CATS_F = ["mpaa_rating", "primary_genre", "origin_group", "distributor_group", "release_weekday"]
XF = D[["log_budget"] + NUMS_F + CATS_F].copy()
MODE_F = {c: XF.loc[XF[c] != "(blank)", c].mode()[0] for c in CATS_F}             # training modes, reused for the test set
for c in CATS_F:
    XF[c] = XF[c].replace("(blank)", MODE_F[c])
XF["distributor_group"] = XF["distributor_group"].replace("Blank", "Other")

pre_lin_f = ColumnTransformer([
    ("budget", make_pipeline(SimpleImputer(strategy="median"), SplineTransformer(n_knots=5, degree=3)), ["log_budget"]),
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMS_F),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATS_F)])
pre_tree_f = ColumnTransformer([("num", "passthrough", ["log_budget"] + NUMS_F),
                                ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False), CATS_F)])
FINAL = {"Logistic regression (final)": make_pipeline(pre_lin_f, LogisticRegression(max_iter=3000, C=0.5)),
         "XGBoost (challenger)": make_pipeline(pre_tree_f, xgb())}
OOF_F = {name: cv_predict(m, XF) for name, m in FINAL.items()}
display(pd.DataFrame([summary_row("Baseline (b)", oof_b, False)] + [summary_row(k, v) for k, v in OOF_F.items()]).set_index("model"))

pr = []
for k in [0.10, 0.20, 0.33, 0.50]:
    for name, s in [("baseline", oof_b), ("logistic (final)", OOF_F["Logistic regression (final)"])]:
        top = top_k(s, k); tp = int(yl[top].sum())
        pr.append({"pick top": f"{k:.0%}", "model": name, "films picked": len(top), "winners picked": tp,
                   "precision": f"{tp / len(top):.0%}", "recall": f"{tp / yl.sum():.0%}"})
print(f"Precision and recall by pick size ({yl.sum():,} winners among {len(yl):,} training films):")
display(pd.DataFrame(pr).set_index(["pick top", "model"]))

In [ ]:
XSF, GF = design(XF, NUMS_F, CATS_F)
full_f, ft_f = fit_table(XSF, GF)
print(f"Final logistic: {len(XSF.columns) - 1} terms; McFadden pseudo-R2 {1 - full_f.llf / null_ll:.3f}; AIC {full_f.aic:.1f} (all candidates: {full.aic:.1f})")
display(ft_f)
odds = pd.DataFrame({"odds ratio": np.exp(full_f.params), "95% low": np.exp(full_f.conf_int()[0]), "95% high": np.exp(full_f.conf_int()[1]),
                     "p": full_f.pvalues}).drop(index="const")
odds["p-value"] = odds.pop("p").map(lambda p: f"{p:.1e}" if p < 0.001 else f"{p:.3f}")
print("Odds of being a winner (numbers per 1 standard deviation; categories vs PG-13 / Drama / US only / Major / Friday). > 1 = more likely a winner.")
with pd.option_context("display.max_rows", 80):
    display(odds.round(2))

**What §4 shows** (run 2026-11-28, cross-validated inside 2000–15; matches `Edwards-A10.ipynb` §18–20 except where noted)

| Model (5-fold CV, 3,708 training films) | AUC (95% range) | Recall @20% | Precision @20% |
|---|---|---|---|
| Random 20% | 0.500 | 20% | 33% |
| Baseline (b): genre × budget tier | 0.609 (0.591–0.625) | 28.4% | 47.0% |
| Five model types, candidate features (4c) | 0.705–0.766 | 34.2–38.3% | 56.6–63.3% |
| **Final logistic regression (4f)** | **0.769 (0.754–0.785)** | **38.9%** | **64.3%** |
| XGBoost challenger, same features | 0.759 (0.742–0.777) | 37.3% | 61.7% |

- **Model choice:** logistic regression, XGBoost, scikit-learn gradient boosting and random forest are statistically tied (their AUC ranges overlap); the single tree is weaker. Logistic regression is the main model because it is at least as accurate, its probabilities are the best calibrated (lowest log loss, 0.526), and every effect can be stated as odds and tested.
- **Fit:** McFadden pseudo-R² 0.192, AIC 3,880. **All 12 features pass the likelihood-ratio test** (p ≤ 0.008) and raise AIC when removed.
- **Biggest effects (odds of a winner):** Horror 2.6×, Documentary 2.5×, G rating 2.2×, Animation 1.9×, Comedy 1.6× vs Drama/PG-13; Wednesday release 1.5× vs Friday; sequel 1.34× and company hit rate 1.33× per SD; more same/neighboring-rating competition 0.78× per SD; R 0.58×, non-US 0.43×, Specialty 0.32× and Other distributors 0.27×, NR 0.06×.
- **Budget is negative once the rest is known (0.44× per SD):** among films with the same distributor, rating and genre, cheaper films more often clear 1.25×. Big budgets look good alone only because they come with majors and sequels.
- **Blanks:** merging blank categories costs almost nothing (logistic 0.766 → 0.764), so the model wasn't leaning on them, but it can no longer use "undocumented film" as a signal.
- **Difference from Tanner's run:** the baseline's recall is 28.4% here vs 28.0% (stable tie-breaking, see §4 intro). Model results are identical.

## 5. Evaluation

Promoted from `Edwards-A10.ipynb` §21 (Tanner, run once on 2026-11-27).

### 5a. Test set 2016–2019 (pre-committed, single test)

Everything is frozen before this runs:
- **Model:** the §4f logistic regression, trained on **all** 2000–2015 films. Challenger: XGBoost, same features.
- **Baseline (b):** genre × budget-tier winner rate from 2000–2015 (smoothed as in §4b).
- **Selection:** the contract picks 20% **each year**, so the main result picks the top 20% within each release year. Pooled top 20% shown as a check.
- **Measures (A9):** recall (share of all test winners in the picks), precision (share of picks that are winners), AUC.
- **Evidence bar (A9):** beat the baseline by **≥ 5 points in recall**, with a **bootstrap 95% CI** (2,000 resamples of test films) that excludes 0.
- Nothing below changes the model. Any change after this point would be tuned on the test set and must be reported that way.

In [ ]:
TEST = films[films["in_population"] & films["year"].between(2016, 2019)].reset_index(drop=True)
yt = TEST["winner"].astype(int).values
years = TEST["year"].astype(int).values
print(f"Test films: {len(TEST):,} (2016-19); winners {yt.mean():.1%}; by year: {TEST.groupby(years).size().to_dict()}")

keep_genres = set(TRN["primary_genre"].value_counts().loc[lambda s: s >= 40].index)


def prep(df):
    """Same transformations as the training data, with every choice learned from the training set."""
    d = df.copy()
    d["mpaa_rating"] = d["mpaa_rating"].replace({"NC-17": "R"})
    d["primary_genre"] = d["primary_genre"].where(d["primary_genre"].isin(keep_genres), "Other")
    for c in ["director", "company"]:
        d[f"{c}_has_history"] = d[f"{c}_prior_hit_rate"].notna().astype(int)
    for c in CATS_F:
        d[c] = d[c].astype(object).fillna("(blank)").astype(str).replace("(blank)", MODE_F[c])
    d["distributor_group"] = d["distributor_group"].replace("Blank", "Other")
    return d[["log_budget"] + NUMS_F + CATS_F]


Xtest = prep(TEST)
p_logit = FINAL["Logistic regression (final)"].fit(XF, yl).predict_proba(Xtest)[:, 1]
p_xgb = FINAL["XGBoost (challenger)"].fit(XF, yl).predict_proba(Xtest)[:, 1]
rates = pd.Series(yl).groupby(g_b.values).agg(["sum", "size"])
g_te = TEST["primary_genre"].fillna("(blank)") + " | " + pd.cut(TEST["fin_budget"], *TIERS).astype(str)
p_base = g_te.map((rates["sum"] + 20 * yl.mean()) / (rates["size"] + 20)).fillna(yl.mean()).values
SCORES_T = {"Baseline (b)": p_base, "Logistic regression (final)": p_logit, "XGBoost (challenger)": p_xgb}


def pick_mask(score, yrs, k=0.2, per_year=True):
    mask = np.zeros(len(score), bool)
    for g in ([np.where(yrs == y)[0] for y in np.unique(yrs)] if per_year else [np.arange(len(score))]):
        mask[g[top_k(score[g], k)]] = True
    return mask


def measures(score, y, yrs, per_year=True):
    w = pick_mask(score, yrs, per_year=per_year)
    return {"recall": y[w].sum() / y.sum(), "precision": y[w].mean(), "AUC": roc_auc_score(y, score)}


rows = []
for per_year in [True, False]:
    for name, s in SCORES_T.items():
        m = measures(s, yt, years, per_year)
        rows.append({"selection": "top 20% each year" if per_year else "top 20% pooled", "model": name, "AUC": round(m["AUC"], 3),
                     "winners caught (recall)": f"{m['recall']:.1%}", "picks that won (precision)": f"{m['precision']:.1%}"})
display(pd.DataFrame(rows).set_index(["selection", "model"]))
print(f"Random 20% would give: recall 20%, precision {yt.mean():.0%}. Best possible recall {min(1, 0.2 / yt.mean()):.0%}.")

In [ ]:
# Evidence standard: difference vs baseline, bootstrap 95% CI (per-year selection)
rng = np.random.default_rng(2016)
diffs = {"Logistic: recall": [], "Logistic: AUC": [], "XGBoost: recall": []}
for _ in range(2000):
    i = rng.integers(0, len(yt), len(yt))
    mb, ml, mx = (measures(SCORES_T[k][i], yt[i], years[i]) for k in SCORES_T)
    diffs["Logistic: recall"].append(ml["recall"] - mb["recall"])
    diffs["Logistic: AUC"].append(ml["AUC"] - mb["AUC"])
    diffs["XGBoost: recall"].append(mx["recall"] - mb["recall"])
mb, ml, mx = (measures(SCORES_T[k], yt, years) for k in SCORES_T)
point = {"Logistic: recall": ml["recall"] - mb["recall"], "Logistic: AUC": ml["AUC"] - mb["AUC"], "XGBoost: recall": mx["recall"] - mb["recall"]}
ev = pd.DataFrame({"difference vs baseline": point, "95% CI low": {k: np.percentile(v, 2.5) for k, v in diffs.items()},
                   "95% CI high": {k: np.percentile(v, 97.5) for k, v in diffs.items()}})
ev.loc[["Logistic: recall", "XGBoost: recall"]] *= 100
ev["meets bar (>= +5 pts, CI above 0)"] = ["yes" if (r["difference vs baseline"] >= 5 and r["95% CI low"] > 0) else ("n/a" if "AUC" in k else "no")
                                           for k, r in ev.iterrows()]
print("Recall rows in percentage points; AUC row in AUC units:")
display(ev.round(3))

In [ ]:
# Calibration, per-year results, and examples (logistic)
TEST["p_win"] = p_logit
cal = TEST.groupby(pd.qcut(TEST["p_win"], 5, labels=["lowest 20%", "2", "3", "4", "highest 20%"]), observed=True).agg(
    films=("film_uid", "size"), predicted_win_rate=("p_win", "mean"), actual_win_rate=("winner", "mean"), median_return=("us_multiple", "median"))
print("Calibration: predicted vs actual winner rate by fifth of predicted chance (test films)")
display(cal.round(2))
yr_rows = []
for y in np.unique(years):
    m = years == y
    r = {"year": y, "films": int(m.sum()), "winners": int(yt[m].sum())}
    for name in ["Baseline (b)", "Logistic regression (final)"]:
        mm = measures(SCORES_T[name][m], yt[m], years[m])
        r[f"{name}: recall"] = f"{mm['recall']:.0%}"; r[f"{name}: precision"] = f"{mm['precision']:.0%}"
    yr_rows.append(r)
display(pd.DataFrame(yr_rows).set_index("year"))
pick = pick_mask(p_logit, years)
show = ["title", "year", "fin_budget", "us_multiple", "distributor_group", "mpaa_rating"]
print("Picks that won the most:")
display(TEST[pick].nlargest(6, "us_multiple")[show])
print("Costliest misses (picked, but under 1.25x):")
display(TEST[pick & ~TEST["winner"].astype(bool).values].nlargest(6, "fin_budget")[show])

**Test results** (2016–19, 855 films, 35.6% winners; the test was run once by Tanner on 2026-11-27 and re-run here on 2026-11-28 with the identical frozen model)

| Top 20% each year | AUC | Winners caught (recall) | Picks that won (precision) |
|---|---|---|---|
| Random | 0.50 | 20% | 36% |
| Baseline (b) | 0.621 | 32.2% | 57.3% |
| **Logistic regression** | **0.831** | **45.1%** | **80.1%** |
| XGBoost challenger | 0.828 | 44.1% | 78.4% |

**Evidence standard (A9: ≥ 5 points over the baseline in winners caught, bootstrap 95% CI above 0): met.**
- **+12.8 points of recall** (95% CI +7.7 to +17.5): the picks caught 45% of all test winners vs 32% for the baseline (best possible 56%).
- **80% of the picks were winners**, vs 57% for the baseline and 36% at random. AUC +0.21 over the baseline (0.17 to 0.25).
- **Calibration is good:** films given a 73% chance won 81% of the time; films given 4% won 2%; the middle fifths match within a few points (the middle fifth is the loosest: 34% predicted, 27% actual).
- **Every test year beats the baseline:** precision 76–90% vs 45–70%; recall 37–49% vs 29–35%. 2019 is the weakest year for recall (37%).
- **Picks that paid off:** *Get Out* (39×), *Searching* (26×), *Halloween* (16×), *Split* (15×), *Lights Out* (14×). **Costliest misses:** big studio sequels below 1.25× in the US: *Transformers: The Last Knight* (0.60×), *Hobbs & Shaw* (0.87×), *Star Trek Beyond*, *Maleficent: Mistress of Evil*, *X-Men: Apocalypse*. Several made most of their money abroad, which the US-only label (A7) doesn't count.
- **Why the test beats cross-validation (AUC 0.831 vs 0.769):** the test years have fewer small films in the data (`Edwards-A10.ipynb` §14b), so easy cases (small, foreign, limited releases) are a bigger share, and the winner rate is a little higher (36% vs 33%). The cross-validated numbers are the more conservative estimate. Whether the screen still helps on harder cases (wide releases only) is §5d.
- **XGBoost ties logistic regression** (within the noise); logistic stays the main model because it is simpler and explainable.
- **Difference from Tanner's run:** baseline recall 32.2% vs his 31.9% and the gain +12.8 vs +13.2 points, from the stable tie-breaking only (§4 intro). XGBoost 0.828 vs his 0.822, from a different XGBoost version (3.4.1 here). The logistic model's numbers are identical.

### Still to come in §5 (Phase 5)

- **5b.** 2020–2023 COVID stress test, reported separately (A5).
- **5c.** Sensitivity: label at 1× and 1.5×, budget floor $100K (A2), model with `_missing` flags.
- **5d.** The harder decision: results among major and mini-major releases only.
- Decision policy and responsible use, value break-even, verdict, Appendix D (independent rebuild of key numbers).

## 6. Deployment

To come.